# Day 9: Advanced Data Wrangling & High-Performance Pandas
> **Programme:** C-DAC PGCP-AI (Post Graduate Certificate Programme in AI)  
> **Course:** Data Analytics  
> **Author:** abs6187 (`23f2000876@ds.study.iitm.ac.in`)  
> **Date:** October 13, 2026  

---

### 🎯 Learning Objectives:
1. **Interactive Jupyter Auto-Suggestion & Help:** Leverage `Tab` auto-completion, `Shift + Tab` parameter tooltips, `?`/`??` introspection, and interactive `suggest_wrangling_help()` utilities.
2. **MultiIndex DataFrame & Hierarchical Slicing:** Construct multi-tiered row and column indices (`from_product`, `from_tuples`) for **North region sales** and national territories, perform precise multidimensional indexing with `pd.IndexSlice` and `.xs()`, reshape dimensions with `stack()` / `unstack()`, and aggregate across index levels.
3. **Vectorised Strings & Regular Expressions:** Leverage the compiled C `.str` accessor to eliminate slow Python loops, clean unstructured text, perform regex extraction with named capture groups (`extract`, `extractall`), and standardize noisy formats with regex backreferences.
4. **Transform Pipeline:** Implement declarative, reproducible data processing pipelines using method chaining and `df.pipe()`, master grouped transformations (`groupby().transform()`) for group-specific Z-scores and imputation, and enforce schema contracts.
5. **Memory Optimisation & Data Type Downcasting:** Profile DataFrame memory footprint at byte-level resolution (`memory_usage(deep=True)`), safely downcast integer and floating-point bitwidths, convert low-cardinality strings to `category`, and build an automated production optimizer delivering $80\%+$ RAM reduction.
6. **High-Performance Query (`df.query` & `pd.eval`):** Master concise query syntax with `@variable` bindings and backtick escaped column names, optimize compound arithmetic with `pd.eval()` by eliminating temporary array allocations, and benchmark multi-threaded NumExpr engine acceleration across 1,000,000 rows.
7. **Critical Interview Traps & Viva Hacks:** Rapid-fire takeaway flashcards covering common interview gotchas on MultiIndex, `.apply()` vs `.str`, `transform()` vs `apply()`, downcasting pitfalls, and `inplace=True`.

---

## 📑 Table of Contents
0. [💡 Jupyter Auto-Suggestion & Inline Help Shortcuts](#0--jupyter-auto-suggestion--inline-help-shortcuts)
1. [Module 1: MultiIndex DataFrame and Hierarchical Slicing (North Region Sales)](#1-multiindex-dataframe-and-hierarchical-slicing-north-region-sales)
   - 1.1 MultiIndex Construction: Regional Sales Performance
   - 1.2 Hierarchical Slicing: North Region Sales via `.loc` & `pd.IndexSlice`
   - 1.3 Cross-Sectional Retrieval (`.xs()`)
   - 1.4 Structural Reshaping (`stack()`, `unstack()`, `swaplevel()`)
   - 1.5 Multi-Level Aggregations (`groupby(level=...)`)
2. [Module 2: Vectorised Strings and Regular Expressions](#2-vectorised-strings-and-regular-expressions)
   - 2.1 The `.str` Accessor vs. Python Loops
   - 2.2 Text Normalization, Splitting & Expanding (`split(expand=True)`)
   - 2.3 Regular Expression Pattern Matching & Extraction with Named Groups
   - 2.4 Multiple Pattern Extraction (`extractall()`)
   - 2.5 Regex Substitution & Backreferences (`replace(regex=True)`)
3. [Module 3: Transform Pipeline](#3-transform-pipeline)
   - 3.1 Declarative Pipelines via Method Chaining and `df.pipe()`
   - 3.2 Modular Functional Steps: Imputation, Outlier Handling, Feature Engineering
   - 3.3 Grouped Transformations: `groupby().transform()` vs `apply()`
   - 3.4 Group-Specific Z-Score Scaling & Group Median Imputation
   - 3.5 End-to-End Composable Pipeline Execution
4. [Module 4: Memory Optimisation and Data Type Downcasting](#4-memory-optimisation-and-data-type-downcasting)
   - 4.1 Internal Memory Architecture & Deep Memory Profiling
   - 4.2 Numeric Downcasting (`int64` $\to$ `int8`/`int16`, `float64` $\to$ `float32`)
   - 4.3 High vs Low Cardinality: Categorical Encoding (`category`)
   - 4.4 Automated Production Downcasting Function (`optimize_dataframe_memory()`)
   - 4.5 Before & After Memory Benchmarks (RAM Reduction $> 80\%$)
5. [Module 5: High-Performance Query (`df.query` & `pd.eval`)](#5-high-performance-query-dfquery--pdeval)
   - 5.1 Query Syntax & Expression Elegance (`df.query()`)
   - 5.2 Dynamic Variable Binding via `@variable`
   - 5.3 Handling Columns with Spaces Using Backticks (`` `Col Name` ``)
   - 5.4 Arithmetic Optimization with `df.eval()` & `pd.eval()`
   - 5.5 Benchmark: Boolean Indexing vs `query(python)` vs `query(numexpr)` on 1,000,000 Rows
6. [Summary & Architectural Scorecard](#6-summary--architectural-scorecard)
7. [Critical Interview Traps & Viva Hacks](#7--critical-interview-traps--viva-hacks-punchy--direct)

---
## 0. 💡 Jupyter Auto-Suggestion & Inline Help Shortcuts

Master these essential IPython & Jupyter shortcuts for rapid coding:

| Action | Shortcut / Syntax | Purpose & Behavior |
| :--- | :---: | :--- |
| **Auto-Suggest / Completion** | `Tab` | Triggers drop-down auto-completion for methods, attributes, and DataFrame column names (e.g., `df_sales.str.<Tab>`). |
| **Docstring / Signature Tooltip** | `Shift + Tab` | Pops up function signature & docstring while typing inside parentheses (press 2x for full docstring, 4x to pin to bottom). |
| **Quick Docstring Inspection** | `object?` (e.g. `pd.IndexSlice?`) | Displays object summary, parameters, and docstrings in a pop-up pager. |
| **Source Code Inspection** | `object??` (e.g. `optimize_dataframe_memory??`) | Displays the full Python source code of any user-defined function. |
| **IPython Quick Reference** | `%quickref` | Displays IPython cheatsheet and available magic commands. |

> **Interactive Helper:** Use the built-in `suggest_wrangling_help(topic)` function below for instant syntax templates and code snippets!

In [1]:
# ==============================================================================
# JUPYTER AUTO-SUGGESTION CONFIGURATION & INTERACTIVE HELP UTILITY
# ==============================================================================
try:
    # Enable greedy auto-completion in IPython/Jupyter for deep chained methods
    get_ipython().run_line_magic('config', 'IPCompleter.greedy = True')
except Exception:
    pass

def suggest_wrangling_help(topic: str = None):
    """
    Interactive auto-suggestion and syntax help utility for Day 9 Data Wrangling.
    Available topics: 'multiindex', 'regex', 'pipeline', 'downcasting', 'query'
    """
    help_db = {
        'multiindex': {
            'Title': 'MultiIndex & Hierarchical Slicing',
            'Key Syntax': "idx = pd.IndexSlice\ndf_sales.loc[idx['North', :, 2026, 'Q1'], idx['Financial', ['Sales_Revenue_Lakhs', 'Profit_Lakhs']]]",
            'Must-Know Method': "df_sales.sort_index(inplace=True)  # MANDATORY before range slicing to prevent errors",
            'Cross-Section': "df_sales.xs(key='Q2', level='Quarter', drop_level=False)",
            'Reshaping': "df.unstack(level='Quarter') | df.stack(level=1) | df.swaplevel('Region', 'Quarter')"
        },
        'regex': {
            'Title': 'Vectorised Strings & Regular Expressions',
            'Named Extract': "df['raw_txn'].str.extract(r'TXN_(?P<Year>\\d{4})-(?P<Region>[A-Z]{2})-(?P<ID>\\d+)')",
            'Multiple Matches': "df['feedback'].str.extractall(r'#(?P<Hashtag>[A-Za-z0-9_]+)')",
            'Expand Split': "df['name'].str.split(' ', n=1, expand=True)",
            'Clean Numeric': "df['raw_amt'].str.replace(',', '', regex=False).astype(float)"
        },
        'pipeline': {
            'Title': 'Declarative Transform Pipeline & GroupBy Transform',
            'Method Chaining': "df_clean = (df.pipe(clean_step).pipe(impute_step, group_col='dept').pipe(validate_schema))",
            'Group Z-Score': "(df[col] - df.groupby('dept')[col].transform('mean')) / df.groupby('dept')[col].transform('std')",
            'Group Impute': "df[col].fillna(df.groupby('dept')[col].transform('median'))",
            'Key Gotcha': "transform() preserves exact N -> N rows; apply() does not!"
        },
        'downcasting': {
            'Title': 'Memory Optimisation & Type Downcasting',
            'Numeric Downcast': "pd.to_numeric(df[col], downcast='integer')  # int64 -> int8/int16/int32\npd.to_numeric(df[col], downcast='float')    # float64 -> float32",
            'Categorical Rule': "df[col].astype('category')  # Use ONLY when unique_count / total_rows < 0.50 (low cardinality)",
            'Profiling': "df.info(memory_usage='deep') | df.memory_usage(deep=True).sum() / (1024**2)"
        },
        'query': {
            'Title': 'High-Performance Query (df.query & pd.eval)',
            'Query Syntax': "df.query('`Col Name` >= @threshold and Region in @target_list', engine='numexpr')",
            'Eval Syntax': "df.eval('Net_Sales = `Gross Sales` * (1 - Discount)', inplace=False)",
            'Key Advantage': "NumExpr compiles C bytecode, chunks CPU cache, eliminates temporary boolean arrays in RAM"
        }
    }

    if not topic or topic.lower() not in help_db:
        print("=" * 70)
        print("  💡 INTERACTIVE AUTO-SUGGESTION HELP: AVAILABLE TOPICS")
        print("=" * 70)
        print("Call suggest_wrangling_help(topic) with one of:")
        for t, info in help_db.items():
            print(f"  • '{t}': {info['Title']}")
        print("=" * 70)
        return

    info = help_db[topic.lower()]
    print("=" * 70)
    print(f"  💡 AUTO-SUGGESTION HELP: {info['Title'].upper()}")
    print("=" * 70)
    for k, v in info.items():
        if k != 'Title':
            print(f"[{k}]:\n  {v}\n")
    print("=" * 70)

# Display interactive suggestion card for MultiIndex as a demonstration
suggest_wrangling_help('multiindex')

  💡 AUTO-SUGGESTION HELP: MULTIINDEX & HIERARCHICAL SLICING
[Key Syntax]:
  idx = pd.IndexSlice
df_sales.loc[idx['North', :, 2026, 'Q1'], idx['Financial', ['Sales_Revenue_Lakhs', 'Profit_Lakhs']]]

[Must-Know Method]:
  df_sales.sort_index(inplace=True)  # MANDATORY before range slicing to prevent errors

[Cross-Section]:
  df_sales.xs(key='Q2', level='Quarter', drop_level=False)

[Reshaping]:
  df.unstack(level='Quarter') | df.stack(level=1) | df.swaplevel('Region', 'Quarter')



---
## 1. MultiIndex DataFrame and Hierarchical Slicing (North Region Sales)

### Theoretical Foundation:
A **MultiIndex** (hierarchical index) allows representation of higher-dimensional data structures ($N > 2$) inside standard two-dimensional tabular DataFrames.

$$\text{Index Tuple} = (\text{Region}, \text{State}, \text{Year}, \text{Quarter})$$

In our primary business scenario, we analyze **Regional Sales Performance** across geographical territories (**North**, **South**, **West**, and **East**), with special operational focus on **North Region Sales** across states (**Delhi**, **Punjab**, and **UP**).

This provides:
1. **$O(1)$ Hash / B-Tree Lookups:** Instant isolation of North Region sales records.
2. **Compact Tabular Storage:** Eliminates redundant string repetition across rows.
3. **Natural Aggregation Boundaries:** Grouping or rolling summaries along any level axis without reshaping.

In [2]:
import pandas as pd
import numpy as np
import time
import re

# Set random seed for reproducibility
rng = np.random.default_rng(seed=42)

# Set Pandas display formatting
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

print("=" * 80)
print("MODULE 1.1: CONSTRUCTING MULTIINDEX (REGIONAL SALES: NORTH REGION FOCUS)")
print("=" * 80)

# 1. Define MultiIndex Levels for Regional Sales Performance
regions = ['North', 'South', 'West', 'East']
states_map = {
    'North': ['Delhi', 'Punjab', 'UP'],
    'South': ['Karnataka', 'Tamil Nadu'],
    'West': ['Maharashtra', 'Gujarat'],
    'East': ['West Bengal', 'Bihar']
}
years = [2025, 2026]
quarters = ['Q1', 'Q2']

# Build row tuples matching realistic sales territory hierarchy
row_tuples = []
for region, states in states_map.items():
    for state in states:
        for year in years:
            for q in quarters:
                row_tuples.append((region, state, year, q))

row_index = pd.MultiIndex.from_tuples(
    row_tuples,
    names=['Region', 'State', 'Year', 'Quarter']
)

# 2. Define Multi-Level Hierarchical Columns (Categories -> Specific Metrics)
col_tuples = [
    ('Financial', 'Sales_Revenue_Lakhs'),
    ('Financial', 'OpEx_Lakhs'),
    ('Financial', 'Profit_Lakhs'),
    ('Operations', 'Units_Sold'),
    ('Operations', 'CSAT_Score')
]

col_index = pd.MultiIndex.from_tuples(
    col_tuples,
    names=['Metric_Type', 'Metric']
)

# 3. Generate Realistic Regional Sales Data Matrix
n_rows = len(row_index)
revenue = np.round(rng.uniform(45.0, 280.0, size=n_rows), 2)
opex = np.round(revenue * rng.uniform(0.55, 0.72, size=n_rows), 2)
profit = np.round(revenue - opex, 2)
units_sold = rng.integers(150, 3200, size=n_rows)
csat = np.round(rng.uniform(82.0, 98.0, size=n_rows), 1)

data_matrix = np.column_stack([revenue, opex, profit, units_sold, csat])

# Instantiate Hierarchical Sales DataFrame
df_sales = pd.DataFrame(data_matrix, index=row_index, columns=col_index)
df_sales[('Operations', 'Units_Sold')] = df_sales[('Operations', 'Units_Sold')].astype(int)

# Sort the MultiIndex for optimal slicing performance
df_sales.sort_index(inplace=True)

print(f"Hierarchical Sales DataFrame Shape: {df_sales.shape}")
print(f"Row Index Levels ({df_sales.index.nlevels}): {df_sales.index.names}")
print(f"Column Index Levels ({df_sales.columns.nlevels}): {df_sales.columns.names}")
print("\nFirst 10 Rows of Regional Sales DataFrame (Showing North Region at Top):")
display(df_sales.head(10))

MODULE 1.1: CONSTRUCTING MULTIINDEX (REGIONAL SALES: NORTH REGION FOCUS)
Hierarchical Sales DataFrame Shape: (36, 5)
Row Index Levels (4): ['Region', 'State', 'Year', 'Quarter']
Column Index Levels (2): ['Metric_Type', 'Metric']

First 10 Rows of Regional Sales DataFrame (Showing North Region at Top):


Metric_Type                               Financial                         Operations           
Metric                          Sales_Revenue_Lakhs OpEx_Lakhs Profit_Lakhs Units_Sold CSAT_Score
Region State       Year Quarter                                                                  
East   Bihar       2025 Q1                   121.57      67.50        54.07       2337       83.4
                        Q2                   132.06      82.44        49.62        641       88.7
                   2026 Q1                   155.35      91.11        64.24       2896       82.7
                        Q2                    89.53      55.46        34.07       1678       89.9
       West Bengal 2025 Q1                    81.26      53.46        27.80       2503       88.9
                        Q2                   205.52     132.38        73.14       2353       92.0
                   2026 Q1                   220.02     141.93        78.09       2275       91.3
                        Q2                   272.36     163.87       108.49       1558       92.4
North  Delhi       2025 Q1                   226.88     129.80        97.08       3177       84.4
                        Q2                   148.14      93.46        54.68       2752       93.1

### 1.2 Hierarchical Slicing: North Region Sales with `.loc` & `pd.IndexSlice`

#### The Slicing Trap:
In a flat DataFrame, slicing uses `df.loc[row_slice, col_slice]`. In a MultiIndex:
- **First Example:** Slicing the outer level directly retrieves **North Region Sales**: `df_sales.loc['North']`.
- To slice **inner levels** (e.g. all `Q1` data across states in North region), a plain `:` syntax inside tuples raises a `SyntaxError`.
- **Solution:** Use `pd.IndexSlice` (`idx = pd.IndexSlice`), which constructs standard Python slice objects for arbitrary levels.

> **CRITICAL RULE:** A MultiIndex **must be sorted** (`df_sales.sort_index()`) prior to range slicing, or Pandas will raise an `UnsortedIndexError`!

In [3]:
print("=" * 80)
print("MODULE 1.2: HIERARCHICAL SLICING (FIRST EXAMPLE: NORTH REGION SALES)")
print("=" * 80)

idx = pd.IndexSlice

# Example A: Slice outer level directly -> North Region Sales
print("--- [A] Slicing Outer Level: North Region Sales ---")
north_region_sales = df_sales.loc['North']
display(north_region_sales)

# Example B: Precise Multi-Level Slice via IndexSlice
# Filter: North Region, All States, Year 2026, Quarter 'Q1'
# Columns: Financial Category (Sales_Revenue_Lakhs and Profit_Lakhs only)
print("\n--- [B] IndexSlice: North Region 2026 Q1 Financial Sales Metrics ---")
north_q1_sales = df_sales.loc[
    idx['North', :, 2026, 'Q1'],
    idx['Financial', ['Sales_Revenue_Lakhs', 'Profit_Lakhs']]
]
display(north_q1_sales)

# Example C: Range Slicing on Hierarchical Index
# Filter: Regions from 'East' to 'North', Year 2025
print("\n--- [C] Range Slice: Regions 'East' to 'North', Year 2025 ---")
range_sales = df_sales.loc[
    idx['East':'North', :, 2025, :],
    :
]
display(range_sales.head(6))

MODULE 1.2: HIERARCHICAL SLICING (FIRST EXAMPLE: NORTH REGION SALES)
--- [A] Slicing Outer Level: North Region Sales ---


Metric_Type                   Financial                         Operations           
Metric              Sales_Revenue_Lakhs OpEx_Lakhs Profit_Lakhs Units_Sold CSAT_Score
State  Year Quarter                                                                  
Delhi  2025 Q1                   226.88     129.80        97.08       3177       84.4
            Q2                   148.14      93.46        54.68       2752       93.1
       2026 Q1                   246.77     145.24       101.53        255       89.1
            Q2                   208.88     138.67        70.21        863       88.1
Punjab 2025 Q1                    67.13      41.91        25.22       2656       86.8
            Q2                   274.27     189.67        84.60        327       92.1
       2026 Q1                   223.87     149.78        74.09       2759       87.8
            Q2                   229.73     138.55        91.18       1008       83.4
UP     2025 Q1                    75.11      51.94        23.17       2948       83.9
            Q2                   150.84     103.60        47.24       1045       97.4
       2026 Q1                   132.14      81.38        50.76       1474       96.5
            Q2                   262.79     157.42       105.37       2168       93.2


--- [B] IndexSlice: North Region 2026 Q1 Financial Sales Metrics ---


Metric_Type                          Financial             
Metric                     Sales_Revenue_Lakhs Profit_Lakhs
Region State  Year Quarter                                 
North  Delhi  2026 Q1                   246.77       101.53
       Punjab 2026 Q1                   223.87        74.09
       UP     2026 Q1                   132.14        50.76


--- [C] Range Slice: Regions 'East' to 'North', Year 2025 ---


Metric_Type                               Financial                         Operations           
Metric                          Sales_Revenue_Lakhs OpEx_Lakhs Profit_Lakhs Units_Sold CSAT_Score
Region State       Year Quarter                                                                  
East   Bihar       2025 Q1                   121.57      67.50        54.07       2337       83.4
                        Q2                   132.06      82.44        49.62        641       88.7
       West Bengal 2025 Q1                    81.26      53.46        27.80       2503       88.9
                        Q2                   205.52     132.38        73.14       2353       92.0
North  Delhi       2025 Q1                   226.88     129.80        97.08       3177       84.4
                        Q2                   148.14      93.46        54.68       2752       93.1

### 1.3 Cross-Sectional Retrieval with `.xs()`

The `.xs()` method provides an expressive, readable API for extracting cross-sections at specific inner levels:
- `df_sales.xs(key, level, axis=0, drop_level=True)`
- Setting `drop_level=False` preserves the full hierarchical coordinate metadata.
- Setting `axis=1` retrieves column-level slices seamlessly.

In [4]:
print("=" * 80)
print("MODULE 1.3: CROSS-SECTIONAL RETRIEVAL VIA .xs()")
print("=" * 80)

# Cross-section across rows: Extract all Q2 Sales records across all regions/states/years
print("--- [A] Row Cross-Section: All Q2 Sales Records (drop_level=False) ---")
q2_cross_section = df_sales.xs(key='Q2', level='Quarter', drop_level=False)
display(q2_cross_section.head(6))

# Cross-section across columns: Extract all Operations metrics across the entire hierarchy
print("\n--- [B] Column Cross-Section: Operations Metrics (axis=1) ---")
ops_cross_section = df_sales.xs(key='Operations', axis=1, level='Metric_Type')
display(ops_cross_section.head(6))

MODULE 1.3: CROSS-SECTIONAL RETRIEVAL VIA .xs()
--- [A] Row Cross-Section: All Q2 Sales Records (drop_level=False) ---


Metric_Type                               Financial                         Operations           
Metric                          Sales_Revenue_Lakhs OpEx_Lakhs Profit_Lakhs Units_Sold CSAT_Score
Region State       Year Quarter                                                                  
East   Bihar       2025 Q2                   132.06      82.44        49.62        641       88.7
                   2026 Q2                    89.53      55.46        34.07       1678       89.9
       West Bengal 2025 Q2                   205.52     132.38        73.14       2353       92.0
                   2026 Q2                   272.36     163.87       108.49       1558       92.4
North  Delhi       2025 Q2                   148.14      93.46        54.68       2752       93.1
                   2026 Q2                   208.88     138.67        70.21        863       88.1


--- [B] Column Cross-Section: Operations Metrics (axis=1) ---


Metric                           Units_Sold  CSAT_Score
Region State       Year Quarter                        
East   Bihar       2025 Q1             2337        83.4
                        Q2              641        88.7
                   2026 Q1             2896        82.7
                        Q2             1678        89.9
       West Bengal 2025 Q1             2503        88.9
                        Q2             2353        92.0

### 1.4 Structural Reshaping & Aggregations

Hierarchical DataFrames can pivot and aggregate fluidly:
- **`unstack(level)`:** Moves an index level into column headers (pivoting from tall to wide).
- **`stack(level)`:** Moves a column level into the row index (pivoting from wide to tall).
- **`swaplevel(i, j)`:** Swaps the order of two index levels.
- **`groupby(level=...)`:** Groups and aggregates along specific hierarchical levels to compare North region performance against other territories.

In [5]:
print("=" * 80)
print("MODULE 1.4: RESHAPING (STACK/UNSTACK) & LEVEL-WISE AGGREGATION")
print("=" * 80)

# 1. Unstack the 'Quarter' level to widen columns for Sales Revenue
print("--- [A] Unstacking 'Quarter' to Columns for Sales_Revenue_Lakhs ---")
revenue_series = df_sales[('Financial', 'Sales_Revenue_Lakhs')]
unstacked_revenue = revenue_series.unstack(level='Quarter')
display(unstacked_revenue.head(6))

# 2. Level Reordering / Swapping
print("\n--- [B] Swapping Levels: ('Quarter', 'Region') to Outer Positions ---")
swapped_df = df_sales.swaplevel('Region', 'Quarter').sort_index()
display(swapped_df.head(4))

# 3. Aggregation across Hierarchical Levels: Compare North vs Other Regions
print("\n--- [C] Hierarchical Aggregation: Total Sales & Profit Comparing North vs Other Regions ---")
level_agg = df_sales.groupby(level=['Region', 'Year']).agg({
    ('Financial', 'Sales_Revenue_Lakhs'): 'sum',
    ('Financial', 'Profit_Lakhs'): 'sum',
    ('Operations', 'Units_Sold'): 'sum',
    ('Operations', 'CSAT_Score'): 'mean'
})
level_agg.columns = ['Total_Sales_Lakhs', 'Total_Profit_Lakhs', 'Total_Units', 'Avg_CSAT']
display(level_agg)

MODULE 1.4: RESHAPING (STACK/UNSTACK) & LEVEL-WISE AGGREGATION
--- [A] Unstacking 'Quarter' to Columns for Sales_Revenue_Lakhs ---


Quarter                      Q1      Q2
Region State       Year                
East   Bihar       2025  121.57  132.06
                   2026  155.35   89.53
       West Bengal 2025   81.26  205.52
                   2026  220.02  272.36
North  Delhi       2025  226.88  148.14
                   2026  246.77  208.88


--- [B] Swapping Levels: ('Quarter', 'Region') to Outer Positions ---


Metric_Type                         Financial                         Operations           
Metric                    Sales_Revenue_Lakhs OpEx_Lakhs Profit_Lakhs Units_Sold CSAT_Score
Quarter State Year Region                                                                  
Q1      Bihar 2025 East                121.57      67.50        54.07       2337       83.4
              2026 East                155.35      91.11        64.24       2896       82.7
        Delhi 2025 North               226.88     129.80        97.08       3177       84.4
              2026 North               246.77     145.24       101.53        255       89.1


--- [C] Hierarchical Aggregation: Total Sales & Profit Comparing North vs Other Regions ---


Total_Sales_Lakhs  Total_Profit_Lakhs  Total_Units   Avg_CSAT
Region Year                                                               
East   2025             540.41              204.63         7834  88.250000
       2026             737.26              284.89         8407  89.075000
North  2025             942.37              331.99        12905  89.616667
       2026            1304.18              493.14         8527  89.683333
South  2025             669.99              242.81         7747  89.850000
       2026             680.53              246.66         7017  91.975000
West   2025             670.12              238.58         5444  88.750000
       2026             737.96              298.57         2685  91.450000

---
## 2. Vectorised Strings and Regular Expressions

### Why the `.str` Accessor is Mandatory:
In raw Python, applying string functions using `df['col'].apply(lambda x: ...)` suffers from three critical bottlenecks:
1. **Python Virtual Machine Overhead:** Iterates through Python objects sequentially, incurring interpreter call overhead on every row.
2. **Missing Value Failures:** A `None` or `np.nan` triggers `TypeError: expected string or bytes-like object`, requiring ugly guard clauses.
3. **Memory Unpacking:** Scalar Python string objects are unboxed from memory arrays.

The Pandas `.str` accessor delegates processing to compiled C/Cython algorithms, natively propagates `NaN` without exceptions, and supports full Regular Expressions (`re`) out of the box.

In [6]:
print("=" * 80)
print("MODULE 2.1: MESSY UNSTRUCTURED TELEMETRY & AUDIT DATA")
print("=" * 80)

raw_customer_records = [
    {
        'raw_id': 'USR-101',
        'raw_name': '   dr. alexander j. o\'connor, ph.d.   ',
        'raw_contact': '+1 (800) 555-0199 ext 402',
        'raw_txn': 'TXN_2026-NA-USA-ORD#88412_USD$14,500.50_STATUS:COMPLETED_FLAG:AUDIT',
        'feedback': 'Amazing experience! Loved the #Speed and #Reliability of delivery. #TopTier'
    },
    {
        'raw_id': 'USR-102',
        'raw_name': '  ms. priya sharma-patel  ',
        'raw_contact': '+91 98765-43210',
        'raw_txn': 'TXN_2025-AP-IND-ORD#19042_INR₹4,20,000.00_STATUS:PENDING_FLAG:NONE',
        'feedback': 'Great support team. Solved #Billing issue within 10 minutes! #SupportWin'
    },
    {
        'raw_id': 'USR-103',
        'raw_name': 'herr DR. KLAUS VON SCHMIDT',
        'raw_contact': '+49 (030) 123456',
        'raw_txn': 'TXN_2026-EM-DEU-ORD#99231_EUR€8,950.00_STATUS:COMPLETED_FLAG:EXPRESS',
        'feedback': 'System was down yesterday. #Downtime #Frustrated with response speed.'
    },
    {
        'raw_id': 'USR-104',
        'raw_name': '  SARAH CONNOR ',
        'raw_contact': '020 7946 0912',
        'raw_txn': 'TXN_2026-EM-GBR-ORD#77312_GBP£3,250.75_STATUS:FLAGGED_FLAG:FRAUD',
        'feedback': 'Refund requested. Inaccurate charges on invoice. #Dispute #Chargeback'
    },
    {
        'raw_id': 'USR-105',
        'raw_name': None,
        'raw_contact': None,
        'raw_txn': 'TXN_2025-NA-CAN-ORD#33109_CAD$6,800.00_STATUS:COMPLETED_FLAG:NONE',
        'feedback': 'Seamless integration with existing APIs. #APIFirst #Productivity'
    }
]

df_raw = pd.DataFrame(raw_customer_records)
print("Raw Unstructured Customer Audit Dataset:")
display(df_raw)

MODULE 2.1: MESSY UNSTRUCTURED TELEMETRY & AUDIT DATA
Raw Unstructured Customer Audit Dataset:


,raw_id,raw_name,raw_contact,raw_txn,feedback
0,USR-101,"dr. alexander j. o'connor, ph.d.",+1 (800) 555-0199 ext 402,"TXN_2026-NA-USA-ORD#88412_USD$14,500.50_STATUS...",Amazing experience! Loved the #Speed and #Reli...
1,USR-102,ms. priya sharma-patel,+91 98765-43210,"TXN_2025-AP-IND-ORD#19042_INR₹4,20,000.00_STAT...",Great support team. Solved #Billing issue with...
2,USR-103,herr DR. KLAUS VON SCHMIDT,+49 (030) 123456,"TXN_2026-EM-DEU-ORD#99231_EUR€8,950.00_STATUS:...",System was down yesterday. #Downtime #Frustrat...
3,USR-104,SARAH CONNOR,020 7946 0912,"TXN_2026-EM-GBR-ORD#77312_GBP£3,250.75_STATUS:...",Refund requested. Inaccurate charges on invoic...
4,USR-105,None,None,"TXN_2025-NA-CAN-ORD#33109_CAD$6,800.00_STATUS:...",Seamless integration with existing APIs. #APIF...


### 2.2 Vectorized Cleaning, Normalization & Splitting

- **`.str.strip()`:** Strips leading/trailing whitespace.
- **`.str.replace()`:** Regular expression substitution.
- **`.str.split(..., expand=True)`:** Splits delimited text directly into individual DataFrame columns without looping!

In [7]:
print("=" * 80)
print("MODULE 2.2: VECTORIZED STRING CLEANING & EXPANSION")
print("=" * 80)

clean_names = (
    df_raw['raw_name']
    .str.strip()
    .str.replace(r'(?i)\b(dr|ms|herr|mr|mrs|ph\.d\.)\b\.?', '', regex=True)
    .str.replace(r'["\']', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)

print("--- [A] Cleaned and Normalized Names ---")
display(pd.DataFrame({'Raw_Name': df_raw['raw_name'], 'Clean_Name': clean_names}))

name_split_df = clean_names.str.split(' ', n=1, expand=True)
name_split_df.columns = ['First_Name', 'Last_Name']

print("\n--- [B] Vectorized Splitting into Structured Columns (expand=True) ---")
display(name_split_df)

MODULE 2.2: VECTORIZED STRING CLEANING & EXPANSION
--- [A] Cleaned and Normalized Names ---


,Raw_Name,Clean_Name
0,"dr. alexander j. o'connor, ph.d.","Alexander J. Oconnor, Ph.D."
1,ms. priya sharma-patel,Priya Sharma-Patel
2,herr DR. KLAUS VON SCHMIDT,Klaus Von Schmidt
3,SARAH CONNOR,Sarah Connor
4,None,None



--- [B] Vectorized Splitting into Structured Columns (expand=True) ---


,First_Name,Last_Name
0,Alexander,"J. Oconnor, Ph.D."
1,Priya,Sharma-Patel
2,Klaus,Von Schmidt
3,Sarah,Connor
4,None,None


### 2.3 Regular Expression Extraction with Named Capture Groups (`.str.extract`)

When data arrives in compound composite codes, **Named Capture Groups (`?P<GroupName>pattern`)** allow Pandas to parse and unpack all constituent fields into structured, typed DataFrame columns in a **single compiled C pass**:

In [8]:
print("=" * 80)
print("MODULE 2.3: REGEX EXTRACTION WITH NAMED CAPTURE GROUPS")
print("=" * 80)

txn_regex = (
    r'TXN_(?P<Year>\d{4})-'
    r'(?P<Region>[A-Z]{2})-'
    r'(?P<Country>[A-Z]{3})-'
    r'ORD#(?P<Order_ID>\d+)_'
    r'(?P<Currency>[A-Z]{3}|[^\d]+)'
    r'(?P<Raw_Amount>[\d,]+(?:\.\d{2})?)_'
    r'STATUS:(?P<Status>[A-Z]+)_'
    r'FLAG:(?P<Audit_Flag>[A-Z]+)'
)

extracted_txns = df_raw['raw_txn'].str.extract(txn_regex)
extracted_txns['Clean_Amount'] = (
    extracted_txns['Raw_Amount']
    .str.replace(',', '', regex=False)
    .astype(float)
)
extracted_txns['Year'] = extracted_txns['Year'].astype(int)

print("Extracted Structured Transaction Ledger:")
display(extracted_txns)

MODULE 2.3: REGEX EXTRACTION WITH NAMED CAPTURE GROUPS
Extracted Structured Transaction Ledger:


,Year,Region,Country,Order_ID,Currency,Raw_Amount,Status,Audit_Flag,Clean_Amount
0,2026,NA,USA,88412,USD$,"14,500.50",COMPLETED,AUDIT,14500.50
1,2025,AP,IND,19042,INR₹,"4,20,000.00",PENDING,NONE,420000.00
2,2026,EM,DEU,99231,EUR€,"8,950.00",COMPLETED,EXPRESS,8950.00
3,2026,EM,GBR,77312,GBP£,"3,250.75",FLAGGED,FRAUD,3250.75
4,2025,NA,CAN,33109,CAD$,"6,800.00",COMPLETED,NONE,6800.00


### 2.4 Multiple Matches (`extractall`) & Regex Backreferences

- **`.str.extractall(regex)`:** Finds **all** occurrences of a pattern within each row and outputs a MultiIndex DataFrame indexed by `(row_index, match_number)`.
- **Regex Substitution with Backreferences:** Reorders and re-formats matching tokens using `\1`, `\2` or `\g<name>` references.

In [9]:
print("=" * 80)
print("MODULE 2.4: MULTIPLE MATCHES (EXTRACTALL) & REGEX BACKREFERENCES")
print("=" * 80)

print("--- [A] Extracting All Hashtags via .str.extractall() ---")
hashtag_pattern = r'#(?P<Hashtag>[A-Za-z0-9_]+)'
extracted_tags = df_raw['feedback'].str.extractall(hashtag_pattern)
display(extracted_tags)

print("\n--- [B] Standardizing Phone Numbers via Regex Backreferences ---")
contact_series = df_raw['raw_contact'].fillna('')

standardized_contact = (
    contact_series
    .str.replace(r'ext\s*\d+', '', regex=True)
    .str.replace(r'[^\d+]', '', regex=True)
)

display(pd.DataFrame({
    'Raw_Contact': df_raw['raw_contact'],
    'Digits_Only': standardized_contact
}))

MODULE 2.4: MULTIPLE MATCHES (EXTRACTALL) & REGEX BACKREFERENCES
--- [A] Extracting All Hashtags via .str.extractall() ---


Hashtag
  match              
0 0             Speed
  1       Reliability
  2           TopTier
1 0           Billing
  1        SupportWin
2 0          Downtime
  1        Frustrated
3 0           Dispute
  1        Chargeback
4 0          APIFirst
  1      Productivity


--- [B] Standardizing Phone Numbers via Regex Backreferences ---


,Raw_Contact,Digits_Only
0,+1 (800) 555-0199 ext 402,+18005550199
1,+91 98765-43210,+919876543210
2,+49 (030) 123456,+49030123456
3,020 7946 0912,02079460912
4,None,


---
## 3. Transform Pipeline

### Declarative Pipeline Architecture:
The modern Pandas paradigm structures data wrangling as a pure functional pipeline with `df.pipe()`.

### `groupby().transform()` vs. `groupby().apply()`
- **`apply()`:** Free-form aggregation or reduction. Can alter the number of rows.
- **`transform()`:** **Guarantees** output has the **exact same length and index** as the input DataFrame ($N \to N$). Enables direct assignment of group-level aggregates back to original rows without joining!

In [10]:
print("=" * 80)
print("MODULE 3.1: SYNTHETIC ENTERPRISE PIPELINE DATASET")
print("=" * 80)

n_records = 20
departments = rng.choice(['Engineering', 'Marketing', 'Sales', 'Finance'], size=n_records)
base_salaries = {
    'Engineering': 110000,
    'Marketing': 75000,
    'Sales': 85000,
    'Finance': 95000
}

salaries = [base_salaries[d] + rng.normal(0, 12000) for d in departments]
sales_volume = [rng.uniform(20000, 180000) if d == 'Sales' else rng.uniform(5000, 40000) for d in departments]

df_pipeline_raw = pd.DataFrame({
    ' Employee_Name ': [f' Emp_{i:03d} ' for i in range(n_records)],
    'DEPT': departments,
    'Annual Salary ($)': np.round(salaries, 2),
    'Quarterly_Sales': np.round(sales_volume, 2),
    'Sick_Days': rng.integers(0, 15, size=n_records)
})

df_pipeline_raw.loc[2, 'Annual Salary ($)'] = np.nan
df_pipeline_raw.loc[7, 'Quarterly_Sales'] = np.nan
df_pipeline_raw.loc[12, 'Annual Salary ($)'] = 850000.0

print("Raw Dataset for Pipeline Processing:")
display(df_pipeline_raw.head(8))

MODULE 3.1: SYNTHETIC ENTERPRISE PIPELINE DATASET
Raw Dataset for Pipeline Processing:


,Employee_Name,DEPT,Annual Salary ($),Quarterly_Sales,Sick_Days
0,Emp_000,Engineering,92350.32,14843.18,4
1,Emp_001,Marketing,62813.05,28079.79,6
2,Emp_002,Sales,NaN,136319.14,14
3,Emp_003,Engineering,120057.52,31902.66,14
4,Emp_004,Sales,108960.77,37238.55,2
5,Emp_005,Engineering,144966.35,37060.41,13
6,Emp_006,Engineering,114972.91,13057.49,1
7,Emp_007,Sales,73125.54,NaN,11


In [11]:
print("=" * 80)
print("MODULE 3.2: DEFINING MODULAR PIPELINE STAGES")
print("=" * 80)

def clean_headers_and_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[\s($)]+', '_', regex=True)
        .str.strip('_')
    )
    str_cols = df.select_dtypes(include='object').columns
    for c in str_cols:
        df[c] = df[c].str.strip()
    return df

def impute_departmental_medians(df: pd.DataFrame, group_col: str, num_cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in num_cols:
        group_median = df.groupby(group_col)[col].transform('median')
        df[col] = df[col].fillna(group_median)
    return df

def winsorize_salary(df: pd.DataFrame, col: str, group_col: str) -> pd.DataFrame:
    df = df.copy()
    p95 = df.groupby(group_col)[col].transform(lambda s: s.quantile(0.95))
    df[col] = np.where(df[col] > p95, p95, df[col])
    return df

def engineer_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dept_mean = df.groupby('dept')['annual_salary'].transform('mean')
    dept_std = df.groupby('dept')['annual_salary'].transform('std')
    df['salary_zscore_dept'] = np.round((df['annual_salary'] - dept_mean) / dept_std, 3)
    
    dept_total_sales = df.groupby('dept')['quarterly_sales'].transform('sum')
    df['dept_sales_pct'] = np.round((df['quarterly_sales'] / dept_total_sales) * 100, 2)
    return df

def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    assert df['annual_salary'].isna().sum() == 0, "Pipeline Error: Unresolved NaNs in annual_salary!"
    assert df['quarterly_sales'].isna().sum() == 0, "Pipeline Error: Unresolved NaNs in quarterly_sales!"
    assert (df['annual_salary'] > 0).all(), "Pipeline Error: Non-positive salary detected!"
    return df

print("Pipeline Stage Functions Defined Successfully.")

MODULE 3.2: DEFINING MODULAR PIPELINE STAGES
Pipeline Stage Functions Defined Successfully.


In [12]:
print("=" * 80)
print("MODULE 3.3: EXECUTING COMPOSEABLE TRANSFORM PIPELINE VIA .pipe()")
print("=" * 80)

df_pipeline_clean = (
    df_pipeline_raw
    .pipe(clean_headers_and_strings)
    .pipe(impute_departmental_medians, group_col='dept', num_cols=['annual_salary', 'quarterly_sales'])
    .pipe(winsorize_salary, col='annual_salary', group_col='dept')
    .pipe(engineer_metrics)
    .pipe(validate_schema)
)

print("Pipeline Execution Completed Cleanly. Verified Zero NaNs.")
display(df_pipeline_clean.head(10))

MODULE 3.3: EXECUTING COMPOSEABLE TRANSFORM PIPELINE VIA .pipe()
Pipeline Execution Completed Cleanly. Verified Zero NaNs.


,employee_name,dept,annual_salary,quarterly_sales,sick_days,salary_zscore_dept,dept_sales_pct
0,Emp_000,Engineering,92350.3200,14843.18,4,-1.383,12.22
1,Emp_001,Marketing,62813.0500,28079.79,6,-0.517,25.28
2,Emp_002,Sales,83096.3800,136319.14,14,0.013,19.12
3,Emp_003,Engineering,120057.5200,31902.66,14,0.394,26.28
4,Emp_004,Sales,102548.2050,37238.55,2,1.806,5.22
5,Emp_005,Engineering,138739.1425,37060.41,13,1.592,30.52
6,Emp_006,Engineering,114972.9100,13057.49,1,0.068,10.75
7,Emp_007,Sales,73125.5400,136319.14,11,-0.906,19.12
8,Emp_008,Finance,69415.4400,24419.84,12,-0.603,33.09
9,Emp_009,Engineering,113212.5400,17982.28,13,-0.045,14.81


---
## 4. Memory Optimisation and Data Type Downcasting

### Memory Architecture of Pandas DataFrames:
| Target DataType | Byte Width | Range / Capacity | Ideal Use Case |
| :--- | :---: | :--- | :--- |
| `int8` | 1 byte | $-128$ to $+127$ | Age, Month, Day of week, Ratings (1–5) |
| `int16` | 2 bytes | $-32,768$ to $+32,767$ | Days of year, Store IDs ($< 30k$) |
| `int32` | 4 bytes | $-2.14 \times 10^9$ to $+2.14 \times 10^9$ | Customer IDs, Order counts |
| `float32` | 4 bytes | $10^{-38}$ to $10^{38}$ (7 decimals) | Prices, Sales, Percentages, Latency |
| `category` | 1–2 bytes code | Up to $65,536$ unique categories | Strings where Unique Count $< 50\%$ of rows |

In [13]:
print("=" * 80)
print("MODULE 4.1: LARGE SCALE DATASET GENERATION (250,000 ROWS)")
print("=" * 80)

n_rows_large = 250_000

df_unoptimized = pd.DataFrame({
    'transaction_id': np.arange(1, n_rows_large + 1, dtype=np.int64),
    'customer_age': rng.integers(18, 85, size=n_rows_large, dtype=np.int64),
    'store_rating': rng.integers(1, 6, size=n_rows_large, dtype=np.int64),
    'region': rng.choice(['North America', 'EMEA', 'APAC', 'LATAM'], size=n_rows_large),
    'device_type': rng.choice(['Mobile iOS', 'Mobile Android', 'Desktop Web', 'Tablet'], size=n_rows_large),
    'payment_method': rng.choice(['Credit Card', 'PayPal', 'Wire Transfer', 'Apple Pay'], size=n_rows_large),
    'order_value': rng.uniform(10.0, 1500.0, size=n_rows_large).astype(np.float64),
    'discount_pct': rng.uniform(0.0, 0.40, size=n_rows_large).astype(np.float64),
    'is_fraud_flag': rng.choice([0, 1], size=n_rows_large, p=[0.99, 0.01]).astype(np.int64)
})

baseline_mem_bytes = df_unoptimized.memory_usage(deep=True).sum()
baseline_mem_mb = baseline_mem_bytes / (1024 ** 2)

print(f"Generated {n_rows_large:,} rows.")
print(f"Baseline Unoptimized Memory Usage: {baseline_mem_mb:.2f} MB")
print("\nMemory Breakdown per Column (MB):")
mem_breakdown = (df_unoptimized.memory_usage(deep=True) / (1024 ** 2)).round(2)
display(mem_breakdown)

MODULE 4.1: LARGE SCALE DATASET GENERATION (250,000 ROWS)


Generated 250,000 rows.
Baseline Unoptimized Memory Usage: 52.81 MB

Memory Breakdown per Column (MB):


Index              0.00
transaction_id     1.91
customer_age       1.91
store_rating       1.91
region            13.23
device_type       14.13
payment_method    14.01
order_value        1.91
discount_pct       1.91
is_fraud_flag      1.91
dtype: float64

In [14]:
print("=" * 80)
print("MODULE 4.2: AUTOMATED MEMORY OPTIMIZATION FUNCTION")
print("=" * 80)

def optimize_dataframe_memory(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
    df_opt = df.copy()

    for col in df_opt.columns:
        col_type = df_opt[col].dtype

        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df_opt[col].min()
            c_max = df_opt[col].max()

            if pd.api.types.is_integer_dtype(col_type) and c_min == 0 and c_max == 1:
                df_opt[col] = df_opt[col].astype(bool)
            elif pd.api.types.is_integer_dtype(col_type):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df_opt[col] = df_opt[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df_opt[col] = df_opt[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df_opt[col] = df_opt[col].astype(np.int32)
                else:
                    df_opt[col] = df_opt[col].astype(np.int64)
            elif pd.api.types.is_float_dtype(col_type):
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df_opt[col] = df_opt[col].astype(np.float32)

        elif col_type == 'object':
            n_unique = df_opt[col].nunique()
            n_total = len(df_opt[col])
            if n_unique / n_total < 0.50:
                df_opt[col] = df_opt[col].astype('category')

    end_mem = df_opt.memory_usage(deep=True).sum() / (1024 ** 2)
    reduction = ((start_mem - end_mem) / start_mem) * 100

    if verbose:
        print("=" * 60)
        print("         MEMORY OPTIMIZATION AUDIT REPORT         ")
        print("=" * 60)
        print(f"Original RAM Footprint:  {start_mem:8.2f} MB")
        print(f"Optimized RAM Footprint: {end_mem:8.2f} MB")
        print(f"RAM Reduction:           {reduction:8.2f} %")
        print("=" * 60)

    return df_opt

df_optimized = optimize_dataframe_memory(df_unoptimized, verbose=True)

print("\nVerifying Statistical Integrity (Original vs. Optimized):")
diff_order_value = abs(df_unoptimized['order_value'].mean() - df_optimized['order_value'].mean())
diff_customer_age = abs(df_unoptimized['customer_age'].mean() - df_optimized['customer_age'].mean())
print(f"Mean Order Value Absolute Difference:  {diff_order_value:.6f}")
print(f"Mean Customer Age Absolute Difference: {diff_customer_age:.6f}")
assert diff_order_value < 1e-3, "Integrity Check Failed: Numerical drift detected!"

MODULE 4.2: AUTOMATED MEMORY OPTIMIZATION FUNCTION


         MEMORY OPTIMIZATION AUDIT REPORT         
Original RAM Footprint:     52.81 MB
Optimized RAM Footprint:     4.29 MB
RAM Reduction:              91.87 %

Verifying Statistical Integrity (Original vs. Optimized):
Mean Order Value Absolute Difference:  0.000032
Mean Customer Age Absolute Difference: 0.000000


---
## 5. High-Performance Query (`df.query` & `pd.eval`)

### The High-Performance Engine: `df.query()` and `pd.eval()` with NumExpr:
1. **No Temporary Arrays:** Compiles expressions into C bytecode; evaluates in CPU cache chunks.
2. **Multi-Threaded Execution:** NumExpr automatically parallelizes array chunks across CPU cores.
3. **Syntax Elegance:** Supports `@variable` bindings and handles column names containing spaces using backticks (`` `Col Name` ``).

In [15]:
print("=" * 80)
print("MODULE 5.1: QUERY SYNTAX MASTERY & DYNAMIC VARIABLE BINDING")
print("=" * 80)

df_sample = df_optimized.head(10000).copy()
df_sample.rename(columns={
    'order_value': 'Order Value (USD)',
    'customer_age': 'Customer Age',
    'device_type': 'Device Type'
}, inplace=True)

min_order_threshold = 750.0
target_regions = ['North America', 'EMEA']
max_age = 45

print("--- [A] Dynamic Query with @ Variables and Backticks ---")
query_result = df_sample.query(
    "`Order Value (USD)` >= @min_order_threshold and "
    "region in @target_regions and "
    "`Customer Age` <= @max_age and "
    "`Device Type` == 'Mobile iOS'"
)

print(f"Total Matches Found: {len(query_result):,}")
display(query_result.head(5))

print("\n--- [B] Computational Evaluation via df.eval() ---")
df_sample = df_sample.eval("Net_Order_Value = `Order Value (USD)` * (1 - discount_pct)")
display(df_sample[['Order Value (USD)', 'discount_pct', 'Net_Order_Value']].head(5))

MODULE 5.1: QUERY SYNTAX MASTERY & DYNAMIC VARIABLE BINDING
--- [A] Dynamic Query with @ Variables and Backticks ---
Total Matches Found: 265


,transaction_id,Customer Age,store_rating,region,Device Type,payment_method,Order Value (USD),discount_pct,is_fraud_flag
1,2,43,5,North America,Mobile iOS,PayPal,1206.444214,0.338672,False
17,18,28,5,EMEA,Mobile iOS,Credit Card,1110.084473,0.369477,False
41,42,26,3,North America,Mobile iOS,Credit Card,845.832214,0.201761,False
53,54,31,1,North America,Mobile iOS,Wire Transfer,1187.326782,0.092663,False
66,67,23,3,EMEA,Mobile iOS,Apple Pay,1014.573853,0.239087,False



--- [B] Computational Evaluation via df.eval() ---


,Order Value (USD),discount_pct,Net_Order_Value
0,1370.002441,0.297789,962.031311
1,1206.444214,0.338672,797.854980
2,791.195068,0.306855,548.413147
3,1179.038940,0.329538,790.500305
4,148.247253,0.003376,147.746765


In [16]:
print("=" * 80)
print("MODULE 5.2: 1,000,000 ROW BENCHMARK (BOOLEAN vs QUERY PYTHON vs QUERY NUMEXPR)")
print("=" * 80)

n_benchmark = 1_000_000
print(f"Generating synthetic benchmark dataset with {n_benchmark:,} rows...")

df_large = pd.DataFrame({
    'A': rng.uniform(0.0, 100.0, size=n_benchmark).astype(np.float32),
    'B': rng.uniform(0.0, 100.0, size=n_benchmark).astype(np.float32),
    'C': rng.integers(1, 1000, size=n_benchmark, dtype=np.int32),
    'Category': rng.choice(['Alpha', 'Beta', 'Gamma', 'Delta'], size=n_benchmark)
})
df_large['Category'] = df_large['Category'].astype('category')

threshold_A = 50.0
threshold_B = 30.0
threshold_C = 250

n_runs = 5
times_bool = []
times_query_py = []
times_query_numexpr = []

for _ in range(n_runs):
    t0 = time.perf_counter()
    res1 = df_large[(df_large['A'] > threshold_A) & (df_large['B'] < threshold_B) & (df_large['C'] >= threshold_C)]
    times_bool.append((time.perf_counter() - t0) * 1000)

for _ in range(n_runs):
    t0 = time.perf_counter()
    res2 = df_large.query("A > @threshold_A and B < @threshold_B and C >= @threshold_C", engine='python')
    times_query_py.append((time.perf_counter() - t0) * 1000)

for _ in range(n_runs):
    t0 = time.perf_counter()
    res3 = df_large.query("A > @threshold_A and B < @threshold_B and C >= @threshold_C", engine='numexpr')
    times_query_numexpr.append((time.perf_counter() - t0) * 1000)

assert len(res1) == len(res2) == len(res3), "Benchmark Error: Result row counts mismatch!"

mean_bool = np.mean(times_bool)
mean_py = np.mean(times_query_py)
mean_numexpr = np.mean(times_query_numexpr)

print("\n" + "=" * 70)
print("       HIGH-PERFORMANCE FILTERING BENCHMARK SCORECARD       ")
print("=" * 70)
print(f"Dataset Size:           {n_benchmark:,} rows")
print(f"Matching Rows:          {len(res1):,} rows ({(len(res1)/n_benchmark)*100:.1f}%)")
print("-" * 70)
print(f"1. Standard Boolean Mask:     {mean_bool:7.2f} ms  (Std: {np.std(times_bool):.2f} ms)")
print(f"2. df.query (Python engine):  {mean_py:7.2f} ms  (Std: {np.std(times_query_py):.2f} ms)")
print(f"3. df.query (NumExpr engine): {mean_numexpr:7.2f} ms  (Std: {np.std(times_query_numexpr):.2f} ms)")
print("-" * 70)
speedup = mean_py / mean_numexpr
print(f"NumExpr vs Python Engine Speedup Factor: {speedup:.2f}x Faster")
print("=" * 70)

MODULE 5.2: 1,000,000 ROW BENCHMARK (BOOLEAN vs QUERY PYTHON vs QUERY NUMEXPR)
Generating synthetic benchmark dataset with 1,000,000 rows...



       HIGH-PERFORMANCE FILTERING BENCHMARK SCORECARD       
Dataset Size:           1,000,000 rows
Matching Rows:          112,564 rows (11.3%)
----------------------------------------------------------------------
1. Standard Boolean Mask:        6.83 ms  (Std: 1.68 ms)
2. df.query (Python engine):    11.00 ms  (Std: 0.80 ms)
3. df.query (NumExpr engine):   13.84 ms  (Std: 0.42 ms)
----------------------------------------------------------------------
NumExpr vs Python Engine Speedup Factor: 0.79x Faster


---
## 6. Summary & Architectural Scorecard

| Module | Core Tool / API | Primary Advantage | When to Use | Key Gotcha |
| :--- | :--- | :--- | :--- | :--- |
| **1. MultiIndex** | `pd.IndexSlice`, `.xs()`, `stack()` / `unstack()` | Represents $N$-dimensional data cubes in 2D with $O(1)$ lookups. | Hierarchical reports (Region $\to$ State $\to$ Year). | Always run `df.sort_index()` before range slicing to prevent errors. |
| **2. Vectorised Strings** | `.str` accessor, Named Regex `(?P<Name>...)` | Compiled C routines; automatic `NaN` propagation; eliminates slow loops. | Parsing unstructured transaction logs, phone numbers, and tags. | Avoid `.apply(lambda x: ...)` for strings; use `.str.extract()` instead. |
| **3. Transform Pipeline** | `df.pipe()`, `groupby().transform()` | Pure functional composability; preserves input index length. | Multi-stage cleaning, group-specific Z-score scaling and imputation. | Use `transform()` instead of `apply()` when broadcasting back to rows. |
| **4. Memory Optimisation** | `pd.to_numeric(downcast=...)`, `category` dtype | Drastically cuts RAM usage ($> 80\%$) and speeds up groupbys. | Datasets with $\ge 100k$ rows or memory-constrained servers. | Ensure unique ratio $< 50\%$ before converting strings to `category`. |
| **5. High-Performance Query** | `df.query(engine='numexpr')`, `pd.eval()` | Avoids temporary RAM array allocations; multi-threaded C execution. | Complex multi-condition filtering on datasets with $> 100k$ rows. | Use backticks (`` `Col Name` ``) for column names containing spaces. |

---
## 7. 🚫 Critical Interview Traps & Viva Hacks (Punchy & Direct)

### ⚠️ Trap 1: The MultiIndex Slicing & Unsorted Index Crash
* **The Trap:** Slicing inner levels of an unsorted MultiIndex (`df.loc[idx[:, 'Q1'], :]`).
* **Why It Fails:** Raises `UnsortedIndexError` or silently causes severe performance degradation.
* **Viva Fix:** Always call `df.sort_index(inplace=True)` before multi-level slicing with `pd.IndexSlice`.

### ⚠️ Trap 2: `transform()` vs. `apply()` in GroupBy
* **The Trap:** Using `apply()` and `transform()` interchangeably to broadcast group metrics back to rows.
* **The Reality:** **NO.** `transform()` strictly guarantees output shape matches input rows ($N \to N$) and aligns with the original index, allowing direct assignment (`df['z'] = ...`). `apply()` can reshape, drop, or return scalar aggregates.
* **Viva Rule:** Use `transform()` whenever broadcasting group statistics back to original rows.

### ⚠️ Trap 3: The `.apply(lambda x: ...)` String Bottleneck
* **The Trap:** Using `.apply(lambda s: s.strip().lower())` or standard Python `re.search()` on string columns.
* **Why It Fails:** Up to $50\times$ slower, creates Python VM interpreter overhead on every row, and crashes on `None`/`np.nan` with `TypeError`.
* **Viva Rule:** Always use compiled `.str` accessor methods (`df['col'].str.strip()`, `.str.extract()`); they natively propagate `NaN` without crashing.

### ⚠️ Trap 4: The Blind `float16` Downcasting Disaster
* **The Trap:** Downcasting `float64` to `float16` to save maximum memory.
* **Why It Fails:** `float16` only has 3–4 decimal digits of precision (causes severe numerical truncation) and lacks hardware SIMD vectorization on most CPUs (runs in slow software emulation).
* **Viva Rule:** Only downcast floats to `float32` (7 decimal digits, full hardware SIMD support).

### ⚠️ Trap 5: The "Categorical Always Saves RAM" Myth
* **The Trap:** Believing `df['col'].astype('category')` always reduces memory.
* **The Reality:** For high-cardinality columns (UUIDs, transaction hashes, unique names where cardinality $> 50\%$), `category` **increases** memory due to dictionary table overhead plus integer codes!
* **Viva Rule:** Only convert to `category` when unique values make up $< 50\%$ (ideally $< 5\%$) of total rows.

### ⚠️ Trap 6: Is `df.query()` Always Faster Than Boolean Indexing?
* **The Trap:** Claiming `df.query()` is always faster than `df[(col1 > x) & (col2 < y)]`.
* **The Reality:** On small DataFrames ($< 10,000$ rows), boolean indexing is faster due to `query()`'s expression parsing/compilation overhead. `query(engine='numexpr')` only wins on large DataFrames ($> 100,000$ rows) where it eliminates temporary intermediate array allocations in RAM.

### ⚠️ Trap 7: The `inplace=True` Fallacy
* **The Trap:** Believing `inplace=True` prevents memory copies and is faster.
* **The Reality:** In almost all Pandas methods (e.g., `dropna`, `reset_index`, `query`), Pandas creates an internal copy anyway and merely reassigns the pointer. `inplace=True` also breaks method chaining pipelines and is deprecated/discouraged in modern Pandas.
* **Viva Rule:** Avoid `inplace=True`; prefer declarative method chaining.

---

### ⚡ Quick-Fire Interview Flashcards
| Interview Question | 1-Line Punchy Answer |
| :--- | :--- |
| **Why use `pd.IndexSlice` over standard slices?** | Standard Python slice `:` is invalid syntax inside tuples for inner MultiIndex levels. |
| **How does `.xs()` differ from `.loc`?** | `.xs()` allows selecting inner levels without specifying outer level wildcards. |
| **What does `expand=True` do in `.str.split()`?** | Returns a multi-column DataFrame instead of a Series of Python lists. |
| **Why is `pipe()` preferred over procedural assignment?** | Enables pure functional method chaining without intermediate mutable global variables. |
| **What engine makes `df.query()` multi-threaded?** | `NumExpr` (evaluates C bytecode in CPU cache chunks). |

---
**Module Completed:** Day 9 Advanced Data Wrangling & High-Performance Pandas  
**Author:** abs6187 (`23f2000876@ds.study.iitm.ac.in`)